In [1]:
from __future__ import annotations

import os
import  operator
from typing import TypedDict, List, Annotated, Literal

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import PydanticOutputParser

from dotenv import load_dotenv
load_dotenv()




True

In [2]:
api_key = os.getenv("OPENAI_API_KEY")

In [3]:
class Task(BaseModel):
    id: int
    title: str

    goal: str = Field(
        ...,
        description="One sentence describing what the reader should be able to do/understand after this section.",
    )
    bullets: List[str] = Field(
        ...,
        min_length=3,
        max_length=5,
        description="3–5 concrete, non-overlapping subpoints to cover in this section.",
    )
    target_words: int = Field(
        ...,
        description="Target word count for this section (120–450).",
    )
    section_type: Literal[
        "intro", "core", "examples", "checklist", "common_mistakes", "conclusion"
    ] = Field(
        ...,
        description="Use 'common_mistakes' exactly once in the plan.",
    )





In [4]:
class Plan(BaseModel):
    blog_title: str
    audience: str = Field(..., description="The target audience for the blog post")
    tone: str = Field(..., description="The tone of the blog post, e.g., practical, crisp etc.")
    task: List[Task]


In [5]:
class State(TypedDict):
    topic: str
    plan: Plan
    ## reducer: results from workers gets concatenated automatically here
    sections: Annotated[List[str], operator.add]
    final: str


In [6]:

llm = ChatOpenAI(model="gpt-4.1-mini", api_key=api_key)

In [7]:
def orchestrator(state: State) -> dict:
    structured_llm = llm.with_structured_output(Plan)

    prompt = [
        SystemMessage(
            content="""
You are a senior technical writer and developer advocate.

Your job is to produce a highly actionable outline for a technical blog post.

Hard requirements:
- Create 5–7 sections (tasks) that fit a technical blog.
- Each section must include:
  1) goal (1 sentence: what the reader can do/understand after the section)
  2) 3–5 concrete, non-overlapping bullets
  3) target word count (120–450)
- Include EXACTLY ONE section with section_type='common_mistakes'.

Make it technical and practical:
- Assume developer-level audience
- Use real engineering structure:
  problem → intuition → approach → implementation → trade-offs → testing → observability → conclusion

Bullet rules:
- Must be actionable (build / test / measure / debug)
- No vague statements like "explain X"

Must include at least one of:
- minimal working example
- edge cases / failure modes
- performance/cost considerations
- security/privacy considerations
- debugging/observability

Ordering:
- intro first
- core concepts next
- common mistakes exactly once
- conclusion/checklist last

Return structured output only.
"""
        ),
        HumanMessage(content=f"Topic: {state['topic']}")
    ]

    # Structured output call (NO parser needed)
    full_plan = structured_llm.invoke(prompt)

    return {"plan": full_plan}

In [8]:
# # 1. Create a parser for your Plan class
# parser = PydanticOutputParser(pydantic_object=Plan)

# def orchestrator(state: State) -> dict:
#     # 2. Get instructions on how the model should format the JSON
#     format_instructions = parser.get_format_instructions()

#     # 3. Use the chat_model (the ChatHuggingFace wrapper)
#     # We add the format instructions to the prompt manually
#     prompt = [
#         SystemMessage(
#             content=f"""
#         You are a senior technical writer and developer advocate. Your job is to produce a highly actionable outline for a technical blog post.
        
#         Hard requirements:
#         - Create 5–7 sections (tasks) that fit a technical blog.
#         - Each section must include:
#         1) goal (1 sentence: what the reader can do/understand after the section)
#         2) 3–5 bullets that are concrete, specific, and non-overlapping
#         3) target word count (120–450)
#         - Include EXACTLY ONE section with section_type='common_mistakes'.

#         Make it technical (not generic):
#         - Assume the reader is a developer; use correct terminology.
#         - Prefer design/engineering flow:
#         problem → intuition → approach → implementation → trade-offs → testing/observability → conclusion.

#         - Bullets must be actionable and testable.
#         - Include at least ONE of:
#         * minimal working example
#         * edge cases / failure modes
#         * performance/cost considerations
#         * security/privacy considerations
#         * debugging tips / observability

#         - Avoid vague bullets.

#         Ordering guidance:
#         - Start with intro/problem framing
#         - Build core concepts first
#         - Include one common mistakes section
#         - End with checklist / next steps

#         {format_instructions}
#         """
#         ),
#         HumanMessage(content=f"Topic: {state['topic']}")
#     ]

#     # 4. Chain the model with the parser
#     chain = chat_model | parser
    
#     # This will return a validated 'Plan' object
#     full_plan = chain.invoke(prompt)
    
#     return {"plan": full_plan}

In [9]:
def fanout(state: State):
    return [
        Send(
            "worker",
            {"task": task, "topic": state["topic"], "plan": state["plan"]},
        )
        for task in state["plan"].task
    ]

In [10]:
def worker(payload: dict) -> dict:

    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    
    bullets = "\n- " + "\n- ".join(task.bullets)

    section_md = llm.invoke([
        SystemMessage(content=
        "You are a senior technical writer and developer advocate. Write ONE section of a technical blog post in Markdown.\n\n"
    
        "Hard constraints:\n"
        "- Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).\n"
        "- Stay close to the Target words (±15%).\n"
        "- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).\n\n"
        "Technical quality bar:\n"
        "- Be precise and implementation-oriented (developers should be able to apply it).\n"
        "- Prefer concrete details over abstractions: APIs, data structures, protocols, and exact terms.\n"
        "- When relevant, include at least one of:\n"
        "  * a small code snippet (minimal, correct, and idiomatic)\n"
        "  * a tiny example input/output\n"
        "  * a checklist of steps\n"
        "  * a diagram described in text (e.g., 'Flow: A -> B -> C')\n"
        "- Explain trade-offs briefly (performance, cost, complexity, reliability).\n"
        "- Call out edge cases / failure modes and what to do about them.\n"
        "- If you mention a best practice, add the 'why' in one sentence.\n\n"
        "Markdown style:\n"
        "- Start with a '## <Section Title>' heading.\n"
        "- Use short paragraphs, bullet lists where helpful, and code fences for code.\n"
        "- Avoid fluff. Avoid marketing language.\n"
        "- If you include code, keep it focused on the bullet being addressed.\n"
        ),
        HumanMessage(
            content=(
                f"Blog Title: {plan.blog_title}\n\n"
                f"Audience: {plan.audience}\n\n"
                f"Tone: {plan.tone}\n\n"
                f"Section: {task.title}\n\n"
                f"Section Type: {task.section_type}\n\n"
                f"Goal: {task.goal}\n\n"
                f"Target words: {task.target_words}\n\n"
                f"Bullets: {bullets}\n\n"
            )
        )
    ]).content

    return {"sections": [section_md]}

In [11]:
from pathlib import Path

def reducer(state: State) -> dict:

    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()
    
    final_md = f"# {title}\n\n{body}\n"

    # Save to file
    filename = "".join(c if c.isalnum() or c in (" ", "_", "-") else "" for c in title)
    filename = filename.strip().lower().replace(" ", "_") + ".md"
    Path(filename).write_text(final_md, encoding="utf-8")

    return {"final": final_md}

In [12]:
graph = StateGraph(State)
graph.add_node("orchestrator", orchestrator)
graph.add_node("worker", worker)
graph.add_node("reducer", reducer)

In [13]:
graph.add_edge(START, "orchestrator")
graph.add_conditional_edges("orchestrator", fanout, ["worker"])
graph.add_edge("worker", "reducer")
graph.add_edge("reducer", END)

app = graph.compile()

In [14]:
output = app.invoke({
    "topic": "Write a blog on Neural Networks.", 
    "sections": []
    })
